# Semana 11: análises exploratórias E7, E8 e E9 (CPU)

Roda os scripts de `analysis/exploratoria/` que seguem `prereg/plano_exploratorio_2.md`. **É exploratório**: não aplica categorias do pré-registro e não substitui nada da rodada confirmatória. Precisa da saída do notebook `semana11_e7_cka.ipynb` (pasta `resultados/semana11/e7_cka` com o manifesto) e das distâncias do E3 (`resultados/semana8/e3/distancias`).

- **E9, índice de fragmentação.** Passo 1: relê os rótulos e grava a máscara de pixels pontuados de cada unidade, conferida com `labels_<conjunto>.npz`. Passo 2: por execução e unidade, as manchas de água prevista (vizinhança de 8 e de 4), o índice (manchas por km² de água prevista), a taxa de erro e a MSP média; por regime e conjunto, o AUROC do índice e o de 1 − MSP média para apontar as unidades ruins, a diferença com IC de 90% e o índice por evento.
- **E8, modelo logístico misto.** Entre os pixels com erro (até 200 por unidade e execução), a chance de o erro não ser sinalizado em função do regime, do domínio, da classe do erro, da faixa de distância à borda e da fração de água, com interceptos aleatórios de unidade, evento e execução, e três sensibilidades.
- **E7, CKA.** Junta os arquivos de CKA com o AUROC_e de cada execução no alvo e confere a média por regime com a rodada confirmatória.

Antes de gravar, cada script confere τ (ou o AUROC_e) contra a rodada confirmatória.

| Célula | O que faz |
|---|---|
| 1 | Instala o `rasterio` e confere o `statsmodels` |
| 2 | Monta o Drive e confere RAM, disco e pastas |
| 3 | Clona o repositório e confere os scripts congelados e o plano |
| 4 | Copia as margens para o disco local e confere pelo manifesto |
| 5 | Baixa os rótulos do Sen1Floods11 e confere pelos hashes da semana 6 |
| 6 | E9, passo 1: máscaras (cerca de 10 minutos; lê as janelas do Drive) |
| 7 | E9, passo 2: análise |
| 8 | E8: modelo misto e sensibilidades (cerca de 1 a 2 horas) |
| 9 | E7: junção do CKA com o AUROC_e |
| 10 | Copia as saídas e o log para `resultados/semana11/e8_e9` |

Use um ambiente com pelo menos 12 GB de RAM; GPU não é necessária. Se alguma célula der erro, pare e guarde a saída inteira.

In [ ]:
# Célula 1 de 10: instalação
!pip install -q "rasterio>=1.4,<1.6"
from importlib.metadata import version
print('rasterio', version('rasterio'))
print('statsmodels', version('statsmodels'))

In [ ]:
# Célula 2 de 10: Drive, RAM, disco e pastas
import os, re, json, time, hashlib, shutil, subprocess, sys, csv, urllib.request
import concurrent.futures as cf
import numpy as np
import psutil

from google.colab import drive
drive.mount('/content/drive')
RAIZ = '/content/drive/MyDrive/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

MARGENS_DRIVE = f'{RAIZ}/resultados/semana6/margens'
CHAVE = f'{RAIZ}/resultados/semana6_chave/chave_codigos.json'
CONFIRMATORIA = f'{RAIZ}/resultados/semana6/analise'
HASHES_S1F11 = f'{RAIZ}/resultados/semana6/entradas_sen1floods11_sha256.csv'
JANELAS = f'{RAIZ}/dados/worldfloods_v2/janelas'
DIST_E3 = f'{RAIZ}/resultados/semana8/e3/distancias'
CKA_DRIVE = f'{RAIZ}/resultados/semana11/e7_cka'
SAIDA_DRIVE = f'{RAIZ}/resultados/semana11/e8_e9'
for caminho in (MARGENS_DRIVE, f'{JANELAS}/label', DIST_E3, CKA_DRIVE):
    assert os.path.isdir(caminho), f'Pasta não encontrada: {caminho}'
for caminho in (CHAVE, f'{CONFIRMATORIA}/resultados.json', HASHES_S1F11, f'{CKA_DRIVE}/manifesto_sha256.csv'):
    assert os.path.isfile(caminho), f'Arquivo não encontrado: {caminho}'
assert not (os.path.isdir(SAIDA_DRIVE) and os.listdir(SAIDA_DRIVE)), (
    'A pasta resultados/semana11/e8_e9 já tem arquivos. Não sobrescrevo: me avise antes de rodar de novo.')

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

ram = psutil.virtual_memory().total / 1e9
disco = shutil.disk_usage('/content')[2] / 1e9
print(f'RAM da sessão: {ram:.1f} GB. Disco local livre: {disco:.0f} GB.')
assert ram >= 12, 'RAM abaixo de 12 GB. Use um ambiente com mais memória.'
assert disco > 30, 'Disco local com menos de 30 GB livres.'

In [ ]:
# Célula 3 de 10: repositório, scripts congelados e plano
REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO = '/content/repo'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()

tabela = open(f'{REPO}/analysis/README.md', encoding='utf-8').read()
registrados = dict(re.findall(r'\| `([a-z_]+\.py)` \| `([0-9a-f]{64})` \|', tabela))
assert set(registrados) == {'analise_congelada.py', 'nucleo.py', 'reamostragem.py', 'categorias.py'}, registrados.keys()
for nome, sha in registrados.items():
    assert sha256_arquivo(f'{REPO}/analysis/{nome}') == sha, f'{nome} é diferente do congelado. Não rode.'
data_plano = subprocess.run(['git', '-C', REPO, 'log', '--diff-filter=A', '-1', '--format=%cI %h', '--',
                             'prereg/plano_exploratorio_2.md'], capture_output=True, text=True, check=True).stdout.strip()
assert data_plano, 'prereg/plano_exploratorio_2.md não está no repositório.'
SHAS = {n: sha256_arquivo(f'{REPO}/analysis/exploratoria/{n}') for n in ('e7_cka.py', 'e8_misto.py', 'e9_fragmentacao.py')}
r = subprocess.run([sys.executable, f'{REPO}/analysis/exploratoria/teste_e7_e8_e9.py'], capture_output=True, text=True)
print(r.stdout[-400:])
assert r.returncode == 0, 'Os testes das funções falharam. Pare e guarde esta saída.'
print(f'Repositório no commit {commit[:7]}. Scripts congelados conferem. Plano adicionado em {data_plano}.')

In [ ]:
# Célula 4 de 10: margens no disco local, conferidas pelo manifesto
LOCAL = '/content/margens'
os.makedirs(LOCAL, exist_ok=True)
manifesto = list(csv.DictReader(open(f'{MARGENS_DRIVE}/manifesto_sha256.csv', newline='')))
assert len(manifesto) == 64, f'O manifesto tem {len(manifesto)} arquivos, esperado 64 (60 execuções e 4 rótulos).'
t0 = time.time()
for linha in manifesto:
    destino = f'{LOCAL}/{linha["arquivo"]}'
    if not (os.path.exists(destino) and os.path.getsize(destino) == int(linha['bytes'])):
        shutil.copy(f'{MARGENS_DRIVE}/{linha["arquivo"]}', destino + '.parcial')
        os.replace(destino + '.parcial', destino)
    assert sha256_arquivo(destino) == linha['sha256'], f'{linha["arquivo"]} diferente do manifesto.'
print(f'{len(manifesto)} arquivos copiados e conferidos pelo manifesto em {(time.time() - t0) / 60:.1f} min.')

In [ ]:
# Célula 5 de 10: rótulos do Sen1Floods11
BUCKET = 'https://storage.googleapis.com/sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/'
LB = '/content/sen1floods11_rotulos'
os.makedirs(LB, exist_ok=True)
hashes = {l['arquivo']: l['sha256'] for l in csv.DictReader(open(HASHES_S1F11, newline=''))}
chaves = [str(c) for conj in ('teste', 'bolivia')
          for c in np.load(f'{LOCAL}/labels_{conj}.npz', allow_pickle=False)['chaves']]
assert len(chaves) == 105 and len(set(chaves)) == 105, len(chaves)

def baixar(chave):
    nome = f'{chave}_LabelHand.tif'
    destino = f'{LB}/{nome}'
    if not (os.path.exists(destino) and sha256_arquivo(destino) == hashes[nome]):
        for tentativa in range(4):
            try:
                urllib.request.urlretrieve(BUCKET + nome, destino + '.parcial')
                os.replace(destino + '.parcial', destino)
                break
            except Exception:
                time.sleep(2 * (tentativa + 1))
    assert sha256_arquivo(destino) == hashes[nome], f'{nome} diferente do hash gravado na semana 6.'
    return nome

with cf.ThreadPoolExecutor(16) as pool:
    feitos = list(pool.map(baixar, chaves))
print(f'{len(feitos)} rótulos do Sen1Floods11 conferidos pelos hashes da semana 6.')

In [ ]:
# Célula 6 de 10: E9, passo 1, máscaras de pixels pontuados (cerca de 10 minutos)
MASC = '/content/e9_mascaras'
LOG = '/content/semana11.log'
if os.path.isdir(MASC):
    shutil.rmtree(MASC)   # saída local de uma tentativa interrompida
cmd = (f'cd {REPO} && set -o pipefail && python analysis/exploratoria/e9_fragmentacao.py mascaras --repo . '
       f'--margens {LOCAL} --s1f11-rotulos {LB} --janelas "{JANELAS}" --saida {MASC} 2>&1 | tee {LOG}')
get_ipython().system(f"bash -c '{cmd}'")
assert all(os.path.isfile(f'{MASC}/mascara_{c}.npz') for c in ('teste', 'bolivia', 't2')), 'O passo 1 do E9 não terminou. Guarde a saída e me mande.'

In [ ]:
# Célula 7 de 10: E9, passo 2, análise
SAIDA = '/content/semana11'
SAIDA_E9 = f'{SAIDA}/e9'
if os.path.isdir(SAIDA_E9):
    shutil.rmtree(SAIDA_E9)   # saída local de uma tentativa interrompida
cmd = (f'cd {REPO} && set -o pipefail && python analysis/exploratoria/e9_fragmentacao.py analise --repo . --margens {LOCAL} '
       f'--chave "{CHAVE}" --confirmatoria "{CONFIRMATORIA}" --mascaras {MASC} --saida {SAIDA_E9} 2>&1 | tee -a {LOG}')
get_ipython().system(f"bash -c '{cmd}'")
assert os.path.isfile(f'{SAIDA_E9}/e9.json'), 'A análise do E9 não terminou. Guarde o log e me mande.'

In [ ]:
# Célula 8 de 10: E8, modelo misto (cerca de 1 a 2 horas). Deixe a aba aberta.
DIST = '/content/e3_distancias'
os.makedirs(DIST, exist_ok=True)
for conj in ('teste', 'bolivia', 't2'):
    shutil.copy(f'{DIST_E3}/dist_{conj}.npz', f'{DIST}/dist_{conj}.npz')
SAIDA_E8 = f'{SAIDA}/e8'
if os.path.isdir(SAIDA_E8):
    shutil.rmtree(SAIDA_E8)
cmd = (f'cd {REPO} && set -o pipefail && python analysis/exploratoria/e8_misto.py --repo . --margens {LOCAL} '
       f'--chave "{CHAVE}" --confirmatoria "{CONFIRMATORIA}" --distancias {DIST} --saida {SAIDA_E8} 2>&1 | tee -a {LOG}')
get_ipython().system(f"bash -c '{cmd}'")
assert os.path.isfile(f'{SAIDA_E8}/e8.json'), 'O E8 não terminou. Guarde o log e me mande.'

In [ ]:
# Célula 9 de 10: E7, junção do CKA com o AUROC_e
CKA = '/content/e7_cka'
os.makedirs(CKA, exist_ok=True)
man = list(csv.DictReader(open(f'{CKA_DRIVE}/manifesto_sha256.csv', newline='')))
assert len(man) == 11, f'O manifesto do E7 tem {len(man)} arquivos, esperado 11.'
for l in man:
    shutil.copy(f'{CKA_DRIVE}/{l["arquivo"]}', f'{CKA}/{l["arquivo"]}')
    assert sha256_arquivo(f'{CKA}/{l["arquivo"]}') == l['sha256'], f'{l["arquivo"]} diferente do manifesto do E7.'
SAIDA_E7 = f'{SAIDA}/e7'
if os.path.isdir(SAIDA_E7):
    shutil.rmtree(SAIDA_E7)
cmd = (f'cd {REPO} && set -o pipefail && python analysis/exploratoria/e7_cka.py --repo . --margens {LOCAL} '
       f'--chave "{CHAVE}" --confirmatoria "{CONFIRMATORIA}" --cka {CKA} --saida {SAIDA_E7} 2>&1 | tee -a {LOG}')
get_ipython().system(f"bash -c '{cmd}'")
assert os.path.isfile(f'{SAIDA_E7}/e7.json'), 'O E7 não terminou. Guarde o log e me mande.'

In [ ]:
# Célula 10 de 10: cópia para o Drive
os.makedirs(SAIDA_DRIVE, exist_ok=True)
for parte in ('e7', 'e8', 'e9'):
    shutil.copytree(f'{SAIDA}/{parte}', f'{SAIDA_DRIVE}/{parte}', dirs_exist_ok=True)
shutil.copy(LOG, f'{SAIDA_DRIVE}/semana11.log')
INFO = {'commit_repositorio': commit, 'plano': 'prereg/plano_exploratorio_2.md', 'plano_adicionado': data_plano,
        'scripts_sha256': SHAS, 'congelados_sha256': registrados,
        'versoes': {q: version(q) for q in ('numpy', 'scipy', 'statsmodels', 'rasterio')},
        'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA_DRIVE}/rodada.json', 'w'), indent=2)
for raiz, _, arquivos in os.walk(SAIDA_DRIVE):
    for a in sorted(arquivos):
        print(os.path.relpath(os.path.join(raiz, a), SAIDA_DRIVE))
print(f'\nSaída copiada para {SAIDA_DRIVE}.')